# Dessin selon des coordonnées géographiques

Proposition de décomposition du problème :

- reconnaître la nécessité d'un choix de projection
- choisir une projection
    - projection naïve
    - projection de Mercator
- calcul des coordonnées extrêmes (*bounding box*)
- calcul de l'échelle (horizontale / verticale)

In [ ]:
import shapefile
from fltk import *
from math import log, tan, pi, radians, degrees

## Projection géographique

Problème : les coordonnées géographiques [WGS84](https://fr.wikipedia.org/wiki/WGS_84) sont exprimées en *degrés* (coordonnées sphériques). Pour les représenter sur une surface plane, il faut associer à chaque couple (longitude, latitude) un couple (abscisse, ordonnée) sur un plan. On appelle cela une [projection cartographique](https://fr.wikipedia.org/wiki/Projection_cartographique). Il existe de nombreuses manières de faire, qui ont toutes différents avantages et inconvénients (déformations aux pôles, non-conservation des aires...). La plus courante est la [*projection de Mercator*](https://fr.wikipedia.org/wiki/Projection_de_Mercator).

In [ ]:
def mercator(point):
    long, lat = point
    x = long
    y = degrees(log(tan(radians(lat) / 2 + pi / 4)))
    return x, y

In [ ]:
# quelques points du contour de la Seine-et-Marne
points = [(2.3923284961351237, 48.335929161584076), (2.5796456483373684, 48.72254039779301), (2.6241630097148376, 49.10056541579139), (3.097802797538821, 49.11080329041811), (3.26191242353348, 48.94707797669175), (3.4474402215212865, 48.848328384354076), (3.4613763255130037, 48.634670779600604), (3.4003188237982123, 48.45958741212868), (3.0995219034980406, 48.3529392805501), (2.8310023496715675, 48.13421017601398), (2.5384374109720103, 48.14073211263637), (2.419849013684824, 48.28321626105787)]
bbox = [2.3923284961351237, 48.12014561527111, 3.559220826259302, 49.11789167125887]

In [ ]:
points_mercator = [mercator(p) for p in points]
points_mercator

In [ ]:
cree_fenetre(500, 500)
polygone(points_mercator)
attend_ev()
ferme_fenetre()

## Calcul des coordonnées à l'écran

Contrainte : il faut garder l'aspect (ratio longueur / largeur)

Stratégie :
- Calculer d'abord le bounding box (point nord-ouest et point sud-est) en coordonnées géographiques
- Chercher le plus grand facteur de zoom possible pour faire tenir la bounding box sur la zone de dessin

Maintenant, supposons que 
- Le point nord-ouest du bounding box est $(x_1, y_1)$, et celui sud-est est $(x_2, y_2)$.
- La zone de dessin est un rectangle de largeur $W$, de hauteur $H$, de coin supérieur gauche $(X_O, Y_O)$ et de coin inférieur droit $(X_O + W, Y_O + H)$.

(Convention : coordonnées géographiques en minuscules, coordonnées écran (pixels) en majuscules.)

### Comment convertir un point géographique $(x, y)$ en un point $(X, Y)$ sur la fenêtre ?


Cela doit être une transformation linéaire (sans déformation) et respectant les proportions (même facteur en abscisses et en ordonnées).  
⚠️⚠️⚠️ **Attention** : les ordonnées sont inversées sur la fenêtre !
$$
    \begin{cases}
    X = ax + B\\
    Y = a(- y) + C.
    \end{cases}
$$

**Contrainte :** Les points correspondant aux coins nord-ouest et sud-est doivent tenir tous les deux dans la zone de dessin. Donc comme $x_1 \leq x_2$ et $y_1 \leq y_2$ on a :
$$
    \begin{cases}
    X_O \leq ax_1 + B \leq ax_2 + B \leq X_O + W\\
    Y_O \leq - ay_2 + C \leq -ay_1 + C \leq Y_O + H
    \end{cases}
$$

Il ne reste plus qu'à résoudre les inéquations pour encadrer $a$, $B$ et $C$ !

#### Encadrement du facteur d'échelle $a$


On a d'une part, pour les abscisses :
$$
X_O - B \leq ax_1 \leq ax_2 \leq X_O + W - B
$$

En réarrangeant on obtient :
$$
\begin{aligned}
ax_2 - ax_1 & \leq (X_O + W - B) - (X_O - B)\\
a(x_2 - x_1) & \leq W\\
a & \leq W / (x_2 - x_1)
\end{aligned}
$$

Pour les ordonnées, on a :
$$
Y_O - C \leq -ay_2 \leq -ay_1 \leq Y_O + H - C
$$

Soit, en réarrangeant :
$$
\begin{aligned}
a(y_2 - y_1) & \leq (Y_O + H - C) - (Y_O - C)\\
a(y_2 - y_1) & \leq H\\
a & \leq H / (y_2 - y_1)
\end{aligned}
$$


On peut donc choisir par exemple $$a = \min \left\{W / (x_2 - x_1), H / (y_2 - y_1)\right\}$$ qui correspond à un zoom maximal.

#### Encadrement des décalages $B$ et $C$


On peut encadrer $B$ et $C$ ainsi :

$$
    \begin{cases}
    X_O - ax_1 \leq B \leq X_O + W - ax_2\\
    Y_O + ay_2 \leq C \leq Y_O + H + ay_1
    \end{cases}
$$

On pourra choisir n'importe quelles valeurs qui conviennent pour placer le dessin où l'on veut (contre l'un des bords, centré...). 


Notons que si $a = W / (x_2 - x_1)$ (dessin ajusté en largeur) alors 
$$
\begin{aligned}
X_O - a x_1 \leq B & \leq X_O + W - W x_2 / (x_2 - x_1)\\
                   & \leq X_O + (W (x_2 - x_1) - W x_2) / (x_2 - x_1)\\
                   & \leq X_O - W x_1 / (x_2 - x_1)\\
                   & \leq X_O - a x_1
\end{aligned}
$$
autrement dit $B$ est forcément égal à $X_O - a x_1$.


De même si $a = H / (y_2 - y_1)$ (dessin ajusté en hauteur) alors 
$$
\begin{aligned}
Y_O + a y_2 \leq C & \leq Y_O + H + H y_1 / (y_2 - y_1)\\
                   & \leq Y_O + (H (y_2 - y_1) + H y_1) / (y_2 - y_1)\\
                   & \leq Y_O + H y_2 / (y_2 - y_1)\\
                   & \leq Y_O + a y_2
\end{aligned}
$$
autrement dit $C$ est forcément égal à $Y_O + a y_2$.

### Implémentation en Python

#### Fonctions de calcul des coordonnées

In [ ]:
def calcule_parametres(bbox, origine, dimensions):
    x_min, y_min, x_max, y_max = bbox
    X_orig, Y_orig = origine
    W, H = dimensions

    # agrandissement max en abscisse
    a_abscisses = W / (x_max - x_min)
    # agrandissement max en ordonnée
    a_ordonnees = H / (y_max - y_min)

    # on suppose qu'on veut un agrandissement max
    # sans dépasser ni en hauteur ni en largeur
    a = min(a_ordonnees, a_abscisses)

    # on cale le dessin dans le coin en haut à gauche
    # TODO : essayer de caler au centre ou sur un autre bord ?
    B = X_orig - a * x_min
    C = Y_orig + a * y_max

    return a, B, C  # à modifier

In [ ]:
def place_point(point, params):
    x, y = point
    a, B, C = params
    X = a * x + B
    Y = - a * y + C
    return X, Y  # à modifier

### Tests

In [ ]:
sf = shapefile.Reader("departements-20180101-shp/departements-20180101")

In [ ]:
departement = sf.shape(76)
sf.record(76)

In [ ]:
departement.bbox

Conversion des coins de la bounding box selon la projection de Mercator.

In [ ]:
coin_so = mercator(departement.bbox[:2])
coin_ne = mercator(departement.bbox[2:])
coin_so, coin_ne

Conversion de l'ensemble des points selon la projection de Mercator.

In [ ]:
points_mercator = [mercator(p) for p in departement.points]
points_mercator[:3]

Calcul des paramètres d'agrandissement et de centrage. Le facteur agrandissement doit être positif. Le signe des autres paramètres dépend de la différence entre les coordonnées de la bounding box et celles de l'origine.

In [ ]:
dimensions = 800, 800
params = calcule_parametres(coin_so + coin_ne, (0, 0), dimensions)
params

Si l'on essaie de placer les points de la bounding box, on devrait constater un ajustement sur au moins deux des bords :

In [ ]:
place_point(coin_so, params), place_point(coin_ne, params)

Quelques résultats de placement de points, pour vérifier (ils doivent tomber entre les points de la bounding box) :

In [ ]:
points_places = [place_point(p, params) for p in points_mercator]
points_places[:3]

Tentative de tracé du département.

In [ ]:
cree_fenetre(800, 800)
polygone(points_places)
attend_ev()
ferme_fenetre()

In [ ]:
sf.close()

#### Structuration en fonctions


##### Objectif 1 : chargement et traitement du shapefile

Construire des objets Python intermédiaires pour faciliter le traitement ultérieur (suggestion : dictionnaire de dictionnaires). Note : lire les données dans le fichier est la partie la plus lente du traitement. Il est souhaitable de ne le faire que le plus rarement possible.

In [ ]:
def decoupe(lst, coupures):
    """
    Découpe une liste en une liste de liste selon les indices de coupure fournis.
    """
    res = []
    fin = 0
    for i in range(1, len(coupures)):
        debut, fin = coupures[i-1], coupures[i]
        res.append(lst[debut:fin])
    res.append(lst[fin:])
    return res

In [ ]:
def charge_departement(sf, index):
    """
    Charge et découpe un département depuis le contenu du shapefile sf.

    Le département est renvoyé sous la forme d'un dictionnaire contenant 
    code INSEE, nom, bbox et liste de polygones.
    """
    record = sf.record(index)
    shape = sf.shape(index)
    bbox = mercator(shape.bbox[:2]) + mercator(shape.bbox[2:])
    points = [mercator(p) for p in shape.points]
    return {
        "code": record[0],
        "nom": record[1],
        "bbox": bbox,
        "polygones": decoupe(points, shape.parts)
    }

In [ ]:
def charge_departements(sf):
    """
    Renvoie un dictionnaire dont les clés sont les codes INSEE des départements
    et dont les valeurs sont des dictionnaires de départements. 
    
    Les points et la bbox sont convertis selon une projection de Mercator.
    """
    res = {}
    for index, record in enumerate(sf.records()):
        code = record[0]
        res[code] = charge_departement(sf, index)
    return res

In [ ]:
import shapefile
with shapefile.Reader("departements-20180101-shp/departements-20180101") as sf:
    deps = charge_departements(sf)

In [ ]:
code = '77'
#deps[code]['nom'], len(deps[code]['polygones'])
deps[code]


##### Objectif 2 : affichage d'un département

De la manière la plus simple et paramétrable possible, créer des fonctions et un programme principal qui affichent un département entier, avec un agrandissement maximal (centrage haut/bas et gauche/droite au choix).

**Variantes :** paramètres permettant de sélectionner le mode de centrage, ajustement du rapport hauteur/largeur de la fenêtre sur la forme à dessiner...

In [ ]:
def dessine_departement(deps, code_insee, origine, dimensions, params=None):
    dep = deps[code_insee]
    if params is None:
        # hypothèse : agrandissement max, ancrage haut-gauche
        params = calcule_parametres(dep["bbox"], origine, dimensions)
    for poly in dep["polygones"]:
        points = [place_point(p, params) for p in poly]
        polygone(points, remplissage='Beige')

In [ ]:
def objectif1(deps, code_insee, W, H):    
    if code_insee in deps:
        cree_fenetre(W, H)
        dessine_departement(deps, code_insee, (0, 0), (W, H))
        attend_ev()
        ferme_fenetre()
    else:
        raise ValueError("département inconnu")

In [ ]:
objectif1(deps, '29', 800, 600)


##### Objectif 3 : affichage de la carte de France

De la manière la plus simple et paramétrable possible, créer des fonctions et un programme principal qui affichent l'ensemble des départements français métropolitains, avec un agrandissement maximal (centrage haut/bas et gauche/droite au choix).

In [ ]:
def calcule_bbox(objets):
    res = [float('inf'), float('inf'), float('-inf'), float('-inf')]
    for objet in objets:
        bbox = objet["bbox"]
        res[0] = min(res[0], bbox[0])
        res[1] = min(res[1], bbox[1])
        res[2] = max(res[2], bbox[2])
        res[3] = max(res[3], bbox[3])
    return res

In [ ]:
def visible(bbox, params, origine, dimensions):
    xmin, ymin, xmax, ymax = bbox
    xmin, ymin = place_point((xmin, ymin), params)
    xmax, ymax = place_point((xmax, ymax), params)
    xo, yo = origine
    w, h = dimensions
    res = not (xmin > xo + w or xmax < xo or ymax > yo + h or ymin < yo)
    # print(f"({xmin}, {ymin}, {xmax}, {ymax}) -> {res}")
    return res

In [ ]:
def dessine_departements(deps, codes, origine, dimensions, params=None):
    if params is None:
        bbox = calcule_bbox([deps[code] for code in codes])
        params = calcule_parametres(bbox, origine, dimensions)
    for code in codes:
        if visible(deps[code]["bbox"], params, origine, dimensions):
            dessine_departement(deps, code, origine, dimensions, params)

In [ ]:
def objectif2(deps, W, H):    
    codes_metro = [code for code in deps if code < '970']
    cree_fenetre(W, H)
    dessine_departements(deps, codes_metro, (0, 0), (W, H))
    attend_ev()
    ferme_fenetre()

In [ ]:
objectif2(deps, 800, 600)


##### Objectif 4 : déplacement et zoom sur la carte

De la manière la plus simple et paramétrable possible, ajouter aux programmes précédents la possibilité de déplacer la carte dans les quatre directions, et celle de zoomer ou dézoomer.

In [ ]:
def traiter_touche(touche, params, W, H):
    if touche == 'plus':
        params[0] *= 2
        params[1] *= 2
        params[2] *= 2
    elif touche == 'minus':
        params[0] /= 2
        params[1] /= 2
        params[2] /= 2
    elif touche == 'Left':
        params[1] -= W/5
    elif touche == 'Right':
        params[1] += W/5
    elif touche == 'Up':
        params[2] -= H/5
    elif touche == 'Down':
        params[2] += H/5
    else:
        return False
    return True

In [ ]:
def objectif3(deps, W, H):
    codes_metro = [code for code in deps if code < '970']
    bbox = calcule_bbox([deps[code] for code in codes_metro])
    params = list(calcule_parametres(bbox, (0, 0), (W, H)))
    
    cree_fenetre(W, H, redimension=True)
    dessine_departements(deps, codes_metro, (0, 0), (W, H), params)

    while True:
        redessine = False
        mise_a_jour()
        ev = donne_ev()
        tev = type_ev(ev)
        if tev == 'Quitte':
            ferme_fenetre()
            break
        elif tev == 'Touche':
            redessine = traiter_touche(touche(ev), params, W, H)
        elif tev == 'Redimension':
            W, H = largeur_fenetre(), hauteur_fenetre()
            redessine = True
        if redessine:
            efface_tout()
            dessine_departements(deps, codes_metro, (0, 0), (W, H), params)

In [ ]:
objectif3(deps, 800, 800)

In [ ]:
ferme_fenetre()


##### Objectif 5 : deux cartes

Dessiner deux cartes de la France métropolitaine côte à côte sur la même fenêtre.

In [ ]:
def objectif5(deps, W, H, marge):
    codes_metro = [code for code in deps if code < '970']
    cree_fenetre(W, H)
    dims = (W/2 - 2*marge, H - marge*2)
    dessine_departements(deps, codes_metro, (marge, marge), dims)   
    dessine_departements(deps, codes_metro, (W/2+marge, marge), dims)
    attend_ev()
    ferme_fenetre()

In [ ]:
objectif5(deps, 1000, 500, 20)